# Auto-Model-Build v2 — thin driver

Thin driver: collect inputs → call an **engine** → render artifacts. No `exec()` of generated code.

Two engines (identical results, different control model):
- **`graph`** — LangGraph orchestrator; HITL via interrupt/resume; set `INTERACTIVE=True` to be prompted at each gate.
- **`linear`** — one-shot procedure; HITL auto-accepts.

Runs offline unless `USE_LLM=True` (then needs `GOOGLE_API_KEY`).

In [1]:
!pip install -r requirements-v2.txt

Ignoring eval_type_backport: markers 'python_version < "3.10"' don't match your environment


In [ ]:
import os
import getpass
import pandas as pd
from dataio import sources

def _menu(title, options, default_idx=0):
    """Simple numbered dropdown (works headless / on CML — no ipywidgets needed)."""
    print(title)
    for i, (key, label) in enumerate(options, 1):
        print(f"  [{i}] {label}")
    raw = input(f"  Choose 1-{len(options)} [default {default_idx+1}]: ").strip()
    idx = (int(raw) - 1) if raw.isdigit() and 1 <= int(raw) <= len(options) else default_idx
    return options[idx][0]

# ── 1) Model Type (hint) ─────────────────────────────────────────────────────
print("=== Model Type ===")
_MODEL_TYPES = [
    ("propensity", "Propensity"), ("cross_sell", "Cross-Sell"), ("up_sell", "Up-Sell"),
    ("churn", "Churn"), ("fraud", "Fraud"), ("credit_risk", "Credit Risk"),
    ("logistic", "Logistic Regression (scorecard)"), ("generic", "Generic"),
]
MODEL_TYPE = _menu("Select model type:", _MODEL_TYPES, default_idx=0)

# ── 2) Data Source ───────────────────────────────────────────────────────────
print("\n=== Data Source ===")
_SOURCE_OPTS = [(s.key, s.label) for s in sources.list_sources()]
SOURCE_KEY = _menu("Select data source:", _SOURCE_OPTS,
                   default_idx=[k for k, _ in _SOURCE_OPTS].index("file"))

# follow-up fields, only what the chosen source needs
_src = sources.get_source(SOURCE_KEY)
_spec_kwargs = {}
if "path" in _src.needs:
    _spec_kwargs["path"] = input("  File / storage path: ").strip()
if "table" in _src.needs:
    _spec_kwargs["table"] = input("  Database.table (e.g. risk_db.cust_features): ").strip()
if "snapshot_col" in _src.needs:
    _spec_kwargs["snapshot_col"] = input("  Snapshot/partition column (optional): ").strip() or None
    _rng = input("  Snapshot range START..END (optional, e.g. 2025-01..2025-06): ").strip()
    if ".." in _rng:
        lo, hi = _rng.split("..", 1)
        _spec_kwargs["snapshot_range"] = (lo.strip(), hi.strip())
if "sql" in _src.needs:
    _spec_kwargs["sql"] = input("  SQL query: ").strip()

SPEC = sources.SourceSpec(**_spec_kwargs)
SOURCE_PATH = _spec_kwargs.get("path") or _spec_kwargs.get("table") or _spec_kwargs.get("sql")

# ── 3) Columns (typed manually — model type is a hint, not a binding) ────────
print("\n=== Columns ===")
TARGET_COL = input("Target column name: ").strip()
DATE_COL   = input("Date/snapshot column name: ").strip()
GROUP_COL  = input("Group/customer-id column name (leave blank to skip): ").strip() or None
OBJECTIVE  = input("Business objective (optional context) [Build CC propensity model]: ").strip() or "Build CC propensity model"

# ── 4) Run settings ─────────────────────────────────────────────────────────
print("\n=== Run Settings ===")
_engine_raw = input("Engine — graph or linear [graph]: ").strip().lower()
ENGINE = _engine_raw if _engine_raw in ("graph", "linear") else "graph"

_interactive_raw = input("Interactive HITL gates? (graph only) y/n [y]: ").strip().lower()
INTERACTIVE = _interactive_raw not in ("n", "no", "false", "0")

_use_llm_raw = input("Enable LLM advisor plane (needs API key)? y/n [n]: ").strip().lower()
USE_LLM = _use_llm_raw in ("y", "yes", "true", "1")

_rounds_raw = input("Search rounds [2]: ").strip()
SEARCH_ROUNDS = int(_rounds_raw) if _rounds_raw.isdigit() else 2

# ── 5) API access via service-account .env (only when LLM is enabled) ─────────────────────────
if USE_LLM:
    from dotenv import load_dotenv
    load_dotenv("/home/cdsw/GenAi_model_dev/.env")

    import os
    print("Credentials:", os.environ.get("GOOGLE_APPLICATION_CREDENTIALS", "NOT SET"))
    print("Project:", os.environ.get("GOOGLE_CLOUD_PROJECT", "NOT SET"))

# ── 6) Load data via the selected source ─────────────────────────────────────
# FileSource is live now; CDP/CML sources (hive/impala/...) raise a clear NotWiredError until Phase 2.
df = sources.load_dataframe(SOURCE_KEY, SPEC)

print(f"\n✓ Loaded: {df.shape[0]:,} rows × {df.shape[1]} cols")
print(f"  model_type={MODEL_TYPE}  source={SOURCE_KEY}")
print(f"  target={TARGET_COL}  date={DATE_COL}  group={GROUP_COL}")
print(f"  engine={ENGINE}  interactive={INTERACTIVE}  use_llm={USE_LLM}  rounds={SEARCH_ROUNDS}")

In [2]:

# 2) Optional: enable the LLM advisor plane (deterministic + cached, temperature=0)
cached_llm = None
if USE_LLM:
    from advisors.llm import build_llm, CachedLLM
    model_id = "gemini-2.5-flash-lite"
    
    cached_llm = CachedLLM(build_llm(model_id), cache_dir="artifacts/runs/_llm_cache", model_id=model_id)
    print("LLM advisor plane ON")
else:
    print("Offline (deterministic) — no API key used")

LLM advisor plane ON


In [ ]:
# ── 3) HITL helpers + run engine ─────────────────────────────────────────────

_W    = 64
_BAR  = "═" * _W
_THIN = "─" * _W

# Stages that expose value overrides; others only get Accept / Regenerate
_OVERRIDE_STAGES = {"model_design"}

_OVERRIDE_PATHS = {
    "primary_metric":          "primary_metric",
    "algorithms":              "search.algorithms",
    "n_trials":                "search.n_trials",
    "imbalance_tier":          "data_strategy.imbalance_tier",
    "correlation_threshold":   "data_strategy.correlation_threshold",
    "psi_threshold":           "data_strategy.psi_threshold",
    "scaling_required":        "data_strategy.scaling_required",
}

def _parse_value(v):
    v = v.strip()
    if "," in v:
        return [x.strip() for x in v.split(",") if x.strip()]
    if v.lower() in ("true", "false"):
        return v.lower() == "true"
    try:   return int(v)
    except ValueError: pass
    try:   return float(v)
    except ValueError: return v

def _render_gate_visual(stage, decisions):
    """Show a chart INLINE at the gate so the reviewer can decide with the data in view.
    Currently: profiling → positive rate per snapshot month (red = drift vs median)."""
    try:
        import matplotlib.pyplot as plt
        import statistics
    except Exception:
        return
    if stage == "profiling":
        pmr = decisions.get("pos_ratio_per_snapshot") or {}
        if len(pmr) >= 2:
            months = list(pmr.keys())
            rates  = [pmr[m] * 100 for m in months]
            med    = statistics.median(rates)
            flag   = [abs(r - med) > max(0.5 * med, 1.0) for r in rates]   # >50% rel & >1% abs
            fig, ax = plt.subplots(figsize=(8.5, 3.2))
            ax.plot(range(len(months)), rates, lw=2, color="#1565C0", zorder=1)
            ax.scatter(range(len(months)), rates,
                       c=["#E53935" if f else "#1565C0" for f in flag], s=40, zorder=2)
            ax.axhline(med, ls="--", lw=1, color="#90A4AE", label=f"median {med:.2f}%")
            for i, r in enumerate(rates):
                ax.annotate(f"{r:.1f}%", (i, r), textcoords="offset points",
                            xytext=(0, 7), ha="center", fontsize=8)
            ax.set_xticks(range(len(months)))
            ax.set_xticklabels(months, rotation=45, ha="right")
            ax.set_ylabel("Positive rate (%)")
            ax.set_title("Positive rate per snapshot  (red = drift vs median; latest months become OOT)")
            ax.set_ylim(0, max(rates) * 1.25)
            ax.legend(fontsize=8)
            plt.tight_layout()
            plt.show()
            if any(flag):
                drifted = [m for m, f in zip(months, flag) if f]
                print(f"  ⚠️  Positive-rate drift in {drifted} — consider this before accepting the window/OOT.")

def _hitl_ui(stage, decisions):
    """Rich HITL prompt: numbered choices, guided override menu, inline chart where useful."""
    can_override = stage in _OVERRIDE_STAGES

    print(f"\n{_BAR}")
    print(f"  🔔  HITL GATE  ─  {stage.upper()}")
    print(_BAR)
    print("  Agent decisions:")
    for k, v in decisions.items():
        if k == "pos_ratio_per_snapshot":
            continue  # rendered as a chart below instead of a long dict
        vstr = str(v)
        if len(vstr) > 50: vstr = vstr[:47] + "..."
        print(f"    •  {k:<28}  {vstr}")
    # show a chart at the gate so you can think about it before deciding
    _render_gate_visual(stage, decisions)
    print(_THIN)
    if can_override:
        print("    [1]  Accept      — proceed with these decisions  (default)")
        print("    [2]  Override    — change specific values interactively")
        print("    [3]  Regenerate  — re-run this step with your feedback")
        prompt = "  Your choice (1 / 2 / 3, or Enter = Accept): "
    else:
        print("    [1]  Accept      — proceed  (default)")
        print("    [3]  Regenerate  — re-run this step with your feedback")
        prompt = "  Your choice (1 / 3, or Enter = Accept): "
    print(_BAR)
    choice = input(prompt).strip() or "1"

    # ── Regenerate ────────────────────────────────────────────────────────
    if choice == "3":
        fb = input("  Describe what to change: ").strip()
        print(f"  ↩  Regenerating with your feedback...")
        return {"action": "regenerate", "instructions": fb, "overrides": {}}

    # ── Override ──────────────────────────────────────────────────────────
    if choice == "2" and can_override:
        params = list(_OVERRIDE_PATHS.keys())
        overrides = {}
        print(f"\n{_THIN}")
        print("  Overridable parameters:")
        for i, k in enumerate(params):
            cur = decisions.get(k, "—")
            print(f"    {chr(97+i)})  {k:<28}  current: {cur}")
        print(_THIN)
        print("  Enter a letter to edit a parameter. Press Enter alone to finish.")
        while True:
            sel = input("  Letter (or Enter to finish): ").strip().lower()
            if not sel:
                break
            idx = ord(sel[0]) - 97
            if not (0 <= idx < len(params)):
                print(f"  ✗  '{sel}' is not a valid option.")
                continue
            key  = params[idx]
            path = _OVERRIDE_PATHS[key]
            cur  = decisions.get(key, "—")
            raw  = input(f"  New value for [{key}]  (current: {cur}): ").strip()
            if not raw:
                print("  (skipped)")
                continue
            val = _parse_value(raw)
            if path.endswith("algorithms") and not isinstance(val, list):
                val = [val]
            overrides[path] = val
            print(f"  ✓  {key} → {val}")
        if overrides:
            print(f"\n  → Overrides: {overrides}")
            return {"action": "override", "overrides": overrides, "instructions": ""}
        print("  → No changes made, accepting.")

    # ── Accept ────────────────────────────────────────────────────────────
    print("  ✓  Accepted — proceeding.\n")
    return {"action": "accept", "overrides": {}, "instructions": ""}

# Adapters for the two engines
def _interactive_resolver(payload):           # graph engine
    return _hitl_ui(payload["stage"], payload["decisions"])

def _hitl_callable(stage, decisions):         # linear engine
    return _hitl_ui(stage, decisions)

# ── Run ───────────────────────────────────────────────────────────────────────
common = dict(
    business_objective=OBJECTIVE, model_type=MODEL_TYPE,
    target_col=TARGET_COL, date_col=DATE_COL, group_col=GROUP_COL,
    cached_llm=cached_llm, source_path=SOURCE_PATH, search_rounds=SEARCH_ROUNDS,
)

if ENGINE == "graph":
    from app import graph
    resolver = _interactive_resolver if INTERACTIVE else None
    out = graph.run_graph(df, resolver=resolver, **common)
    result, model, prepared, summary = (
        out["result"], out["best_model"], out["prepared"], out["summary"])
    gates_info = f"gates seen: {out['gates_seen']}"
else:
    from app import orchestrator
    hitl_mode = _hitl_callable if INTERACTIVE else "auto"
    result, model, prepared, summary = orchestrator.run(
        df, hitl_mode=hitl_mode, **common)
    gates_info = "interactive HITL" if INTERACTIVE else "auto (no prompts)"

print(f"\n{_BAR}")
print(f"  ✅  RUN COMPLETE  —  {ENGINE} engine  ({gates_info})")
print(_THIN)
print(f"  Model type       :  {MODEL_TYPE}")
print(f"  Best algorithm   :  {result.best_algorithm}")
print(f"  Primary metric   :  {result.primary_metric}")
print(f"  Trials run       :  {result.n_trials_run}")
print(f"  Artifacts at     :  {result.run_dir}")
print(_BAR)

In [ ]:
# ── Data Profiling: Positive rate per snapshot month ─────────────────────────
# Reads data_profile.json (written during profiling). Shows a TABLE (readable with many
# snapshots) + a line chart. OOT hold-out months are shaded; median line makes drift obvious.
import json, statistics
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

_prof = {}
_pp = Path(result.run_dir) / "data_profile.json"
if _pp.exists():
    _prof = json.loads(_pp.read_text(encoding="utf-8"))
_temporal = _prof.get("temporal", {}) or {}
pmr = _temporal.get("pos_ratio_per_snapshot", {}) or {}
rps = _temporal.get("rows_per_snapshot", {}) or {}

if pmr:
    months  = list(pmr.keys())
    rates   = [pmr[m] * 100 for m in months]
    oot_set = set(prepared.split_report.oot_range or [])
    med     = statistics.median(rates)

    # ── table (stays readable even with many snapshots) ──────────────────────
    prof_tbl = pd.DataFrame({
        "Snapshot":      months,
        "Rows":          [f"{rps.get(m, 0):,}" for m in months],
        "Positive rate": [f"{r:.2f}%" for r in rates],
        "Set":           ["OOT (hold-out)" if m in oot_set else "Train" for m in months],
    })
    print("Positive rate per snapshot month:")
    print(prof_tbl.to_string(index=False))
    print(f"\n  median positive rate = {med:.2f}%")

    # ── line chart ──────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(9, 3.6))
    ax.plot(range(len(months)), rates, marker="o", lw=2, color="#1565C0", label="positive rate")
    ax.axhline(med, ls="--", lw=1, color="#90A4AE", label=f"median {med:.2f}%")
    for i, m in enumerate(months):
        if m in oot_set:
            ax.axvspan(i - 0.5, i + 0.5, color="#FFE0B2", alpha=0.6)
    for i, r in enumerate(rates):
        ax.annotate(f"{r:.1f}%", (i, r), textcoords="offset points", xytext=(0, 7),
                    ha="center", fontsize=8)
    ax.set_xticks(range(len(months)))
    ax.set_xticklabels(months, rotation=45, ha="right")
    ax.set_ylabel("Positive rate (%)")
    ax.set_title("Positive Rate per Snapshot Month  (orange band = OOT hold-out)")
    ax.set_ylim(0, max(rates) * 1.25)
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print("No per-snapshot positive rate available (needs a date column + binary target).")

In [ ]:
# ── 4) Results Dashboard ─────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import numpy as np

_W    = 64
_BAR  = "═" * _W
_THIN = "─" * _W

# real-name helpers
rev   = (result.column_alias_map or {}).get("reverse", {})
real1 = lambda n:     rev.get(n, n)
realN = lambda names: [rev.get(n, n) for n in (names or [])]

sel = prepared.selection_report
sr  = prepared.split_report

# iv_table records use keys "IV" + "strength"; read case-insensitively so it can't silently 0-out
_iv  = lambda r: r.get("IV", r.get("iv", 0))            # IV value
_ivb = lambda r: r.get("strength", r.get("iv_band", "")) # IV band/strength label

# Resolve primary_metric key in oot_metrics (config stores "PR_AUC", oot dict uses "pr_auc")
_sample_oot = next(
    (r.get("oot_metrics", {}) for r in result.leaderboard if r.get("oot_metrics")), {}
)
_pm_key = next(
    (k for k in _sample_oot if k.lower() == result.primary_metric.lower()),
    result.primary_metric.lower()          # fallback: lowercase
)
_pm_col = f"OOT {_pm_key}"                # column name used throughout leaderboard DataFrame

def _sec(n, title):
    print(f"\n{_BAR}\n  {n}  {title}\n{_THIN}")

# ══════════════════════════════════════════════════════════════════════════════
# 1. DATA SPLIT
# ══════════════════════════════════════════════════════════════════════════════
_sec("①", "DATA SPLIT")
split_df = pd.DataFrame([
    {"Set": "Train",
     "Rows": f"{sr.train_rows:,}",
     "Snapshots": " → ".join(sr.train_range) if sr.train_range else "—",
     "Positive ratio": f"{sr.train_pos_ratio:.2%}" if sr.train_pos_ratio is not None else "—"},
    {"Set": "OOT (hold-out)",
     "Rows": f"{sr.oot_rows:,}",
     "Snapshots": " → ".join(sr.oot_range) if sr.oot_range else "—",
     "Positive ratio": f"{sr.oot_pos_ratio:.2%}" if sr.oot_pos_ratio is not None else "—"},
])
print(split_df.to_string(index=False))

# ══════════════════════════════════════════════════════════════════════════════
# 2. FEATURE SELECTION FUNNEL
# ══════════════════════════════════════════════════════════════════════════════
_sec("②", "FEATURE SELECTION FUNNEL")
funnel_stages = [
    ("Input features",                       sel.n_in),
    ("After hygiene (constant/missing/card)", sel.n_after_hygiene),
    ("After univariate IV filter",            sel.n_after_univariate),
    ("After redundancy / corr pruning",       sel.n_after_redundancy),
    ("Final selected",                        sel.n_selected),
]
mx = sel.n_in or 1
for label, n in funnel_stages:
    bar_len = int(36 * n / mx)
    removed = sel.n_in - n
    marker  = "  ◀ FINAL" if label == "Final selected" else (f"  (removed {removed} total)" if removed else "")
    print(f"  {label:<44} {n:>4}  {'█' * bar_len}{marker}")

print(f"\n  Dropped by reason:")
drop_cats = [
    ("Constant / zero-variance",     sel.dropped_constant),
    ("High missing rate",            sel.dropped_high_missing),
    ("High cardinality categorical", sel.dropped_high_cardinality),
    ("Low IV (below threshold)",     sel.dropped_low_iv),
    ("Redundant (corr cluster)",     sel.dropped_redundant),
]
for label, lst in drop_cats:
    if not lst:
        continue
    names = ", ".join(realN(lst[:6])) + (f" … +{len(lst)-6} more" if len(lst) > 6 else "")
    print(f"    • {label:<36} ({len(lst):>3})  {names}")
if sel.suspicious_high_iv:
    names = ", ".join(realN(sel.suspicious_high_iv[:6]))
    print(f"    ⚠️  Leakage suspects (high IV, flagged not dropped)  "
          f"({len(sel.suspicious_high_iv)})  {names}")

# ══════════════════════════════════════════════════════════════════════════════
# 3. IV TABLE — SELECTED FEATURES
# ══════════════════════════════════════════════════════════════════════════════
_sec("③", "INFORMATION VALUE — SELECTED FEATURES  (real names, top 20)")
selected_set = set(sel.selected_features)
iv_rows = sorted(
    [r for r in sel.iv_table if r["feature"] in selected_set],
    key=_iv, reverse=True
)[:20]
if iv_rows:
    iv_df = pd.DataFrame([
        {
            "Feature":  real1(r["feature"]),
            "IV":       round(_iv(r), 4),
            "IV band":  _ivb(r),
            "Leakage?": "⚠️" if r["feature"] in sel.suspicious_high_iv else "",
        }
        for r in iv_rows
    ])
    print(iv_df.to_string(index=False))

    fig, ax = plt.subplots(figsize=(9, max(3, len(iv_df) * 0.36 + 1)))
    colors_iv = ["#E53935" if r["feature"] in sel.suspicious_high_iv else "#1565C0"
                 for r in iv_rows]
    ax.barh(iv_df["Feature"][::-1], iv_df["IV"][::-1], color=colors_iv[::-1])
    ax.set_xlabel("Information Value (IV)")
    ax.set_title("IV — Selected Features  (red = leakage suspect)")
    ax.axvline(0.02, color="orange", lw=1, ls="--", label="weak (0.02)")
    ax.axvline(0.1,  color="green",  lw=1, ls="--", label="medium (0.10)")
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print("  (no IV data available)")

# ══════════════════════════════════════════════════════════════════════════════
# 4. MODEL LEADERBOARD
# ══════════════════════════════════════════════════════════════════════════════
_sec("④", "MODEL LEADERBOARD — ALL TRIALS")

# Deduplicate: keep one row per (algorithm, cv_score) pair with best OOT primary metric
_seen: dict = {}
unique_rows = []
for r in result.leaderboard:
    key = (r["algorithm"], round(r.get("cv_score", 0), 5))
    oot_pm = r.get("oot_metrics", {}).get(_pm_key, 0)
    if key not in _seen or oot_pm > _seen[key]:
        _seen[key] = oot_pm
        unique_rows.append(r)

oot_keys = sorted({k for r in unique_rows for k in r.get("oot_metrics", {})})

# Best OOT primary metric per algorithm (for the ★ marker)
_best_oot_per_algo = {}
for r in unique_rows:
    a = r["algorithm"]
    v = r.get("oot_metrics", {}).get(_pm_key, 0)
    _best_oot_per_algo[a] = max(_best_oot_per_algo.get(a, 0), v)

lb_records = []
for r in unique_rows:
    oot_pm_val = r.get("oot_metrics", {}).get(_pm_key, 0)
    is_best = (r["algorithm"] == result.best_algorithm and
               oot_pm_val == _best_oot_per_algo.get(result.best_algorithm, 0))
    rec = {
        "★":         "★" if is_best else "",
        "Algorithm": r["algorithm"],
        "CV score":  round(r.get("cv_score", 0), 4),
    }
    for k in oot_keys:
        rec[f"OOT {k}"] = round(r.get("oot_metrics", {}).get(k, float("nan")), 4)
    lb_records.append(rec)

lb_display = (pd.DataFrame(lb_records)
                .sort_values(_pm_col, ascending=False)
                .reset_index(drop=True))
print(lb_display.to_string(index=False))

# Best-per-algorithm bar chart
best_per_algo = (lb_display.groupby("Algorithm")[_pm_col].max()
                           .sort_values(ascending=False))
fig, ax = plt.subplots(figsize=(8, max(3, len(best_per_algo) * 0.55 + 1)))
colors_lb = ["#2E7D32" if a == result.best_algorithm else "#90A4AE"
             for a in best_per_algo.index]
bars = ax.barh(best_per_algo.index[::-1], best_per_algo.values[::-1], color=colors_lb[::-1])
ax.bar_label(bars, fmt="%.4f", padding=5, fontsize=9)
ax.set_xlabel(_pm_col)
ax.set_title(f"Best {_pm_col} per Algorithm  (★ = winner)")
ax.set_xlim(0, best_per_algo.max() * 1.18)
plt.tight_layout()
plt.show()

# ══════════════════════════════════════════════════════════════════════════════
# 5. FEATURE IMPORTANCE — BEST MODEL
# ══════════════════════════════════════════════════════════════════════════════
_sec("⑤", f"FEATURE IMPORTANCE — {result.best_algorithm}  (top 20, real names)")
fi = None
if hasattr(model, "feature_importances_"):
    fi = model.feature_importances_
elif hasattr(model, "coef_"):
    fi = np.abs(model.coef_[0] if model.coef_.ndim > 1 else model.coef_)

fnames = realN(prepared.feature_names) if prepared.feature_names else []
if fi is not None and fnames:
    fi_norm = fi / (fi.sum() or 1)
    fi_df = (pd.DataFrame({"Feature": fnames, "Importance": fi_norm})
               .sort_values("Importance", ascending=False).head(20))
    print(fi_df.to_string(index=False))

    fig, ax = plt.subplots(figsize=(9, max(4, len(fi_df) * 0.38 + 1)))
    cmap = plt.cm.Blues_r(np.linspace(0.15, 0.65, len(fi_df)))
    bars = ax.barh(fi_df["Feature"][::-1], fi_df["Importance"][::-1], color=cmap[::-1])
    ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
    ax.set_xlabel("Relative importance (normalised)")
    ax.set_title(f"Feature Importance — {result.best_algorithm}")
    plt.tight_layout()
    plt.show()
else:
    print("  (feature importance not available for this model type)")

# ══════════════════════════════════════════════════════════════════════════════
# 6. OOT METRICS — BEST MODEL
# ══════════════════════════════════════════════════════════════════════════════
_sec("⑥", f"OOT METRICS — BEST MODEL  ({result.best_algorithm})")
best_row = next(
    (r for r in result.leaderboard
     if r.get("algorithm") == result.best_algorithm
     and r.get("oot_metrics", {}).get(_pm_key, 0) == _best_oot_per_algo.get(result.best_algorithm, 0)),
    {}
)
oot = best_row.get("oot_metrics", {})
if oot:
    m_df = pd.DataFrame([{"Metric": k, "Value": f"{v:.4f}",
                           "Bar": "█" * int(v * 30)} for k, v in oot.items()])
    print(m_df.to_string(index=False))

    fig, ax = plt.subplots(figsize=(7, 3.2))
    bar_colors = ["#1976D2" if k == _pm_key else "#64B5F6" for k in oot]
    rects = ax.bar(list(oot.keys()), list(oot.values()), color=bar_colors)
    ax.bar_label(rects, fmt="%.4f", padding=3, fontsize=9)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel("Score")
    ax.set_title(f"OOT Metrics — {result.best_algorithm}  (primary = {_pm_key})")
    legend = [mpatches.Patch(color="#1976D2", label=f"Primary ({_pm_key})"),
              mpatches.Patch(color="#64B5F6", label="Other metrics")]
    ax.legend(handles=legend, fontsize=8)
    plt.tight_layout()
    plt.show()

# ══════════════════════════════════════════════════════════════════════════════
# 7. THRESHOLDS + NARRATIVE
# ══════════════════════════════════════════════════════════════════════════════
_sec("⑦", "DECISION THRESHOLDS")
print(f"  F1-optimal threshold  :  {result.optimal_threshold:.4f}")
for k, v in (result.business_thresholds or {}).items():
    print(f"  {k:<32}  {v}")

_sec("⑧", "MODEL NARRATIVE")
print(summary)

print(f"\n{_BAR}")
print(f"  Artifacts  →  {result.run_dir}")
print(_BAR)

In [ ]:
# ── 4b) Validation curves (auto-saved to artifacts/<run>/charts/) ────────────
# Generated by pipeline/charts.py during finalize and logged as PNGs (each PNG has a one-line
# explanation printed on it). ALL charts (incl. IV, importance, leaderboard, OOT, posrate) are in
# the same folder for sharing / MRM. Large-data-safe: curves use the OOT vector with row caps.
from pathlib import Path
from IPython.display import Image, Markdown, display

_charts = Path(result.run_dir) / "charts"
print(f"All charts logged to: {_charts}")
print("Files:", sorted(p.name for p in _charts.glob('*')) if _charts.exists() else "—")

# (title, what it tells you) — the same one-liner is also printed on each PNG
_show = [
    ("gains_lift.png",      "Gains / Lift",            "How many real positives you capture by targeting top-scored customers — steeper = better."),
    ("roc_pr.png",          "ROC & Precision-Recall",  "ROC = ranking ability (AUC); PR = precision/recall trade-off when positives are rare."),
    ("ks_curve.png",        "KS curve",                "Biggest gap between positive vs negative score distributions — higher = cleaner separation."),
    ("calibration.png",     "Calibration",             "Are predicted probabilities trustworthy? On the diagonal = a predicted 30% really converts ~30%."),
    ("score_psi.png",       "Score PSI (stability)",   "Did the model's SCORE distribution shift train->OOT? >0.25 = unstable population."),
    ("csi_per_feature.png", "Feature stability (CSI)", "How much each FEATURE shifted train->OOT; >0.25 (red) = unstable."),
    ("woe_trends.png",      "WoE trends",              "How a feature's odds change across its value bins; smooth/monotonic = well-behaved."),
]
for _file, _title, _desc in _show:
    _p = _charts / _file
    if _p.exists():
        display(Markdown(f"**{_title}** — _{_desc}_"))
        display(Image(filename=str(_p)))
    else:
        print(f"  (skipped {_file} — not generated)")

In [5]:
# 5) Score new data with the saved bundle (no retraining, no LLM)
import joblib, numpy as np
bundle = joblib.load(f"{result.run_dir}/model_bundle.joblib")
def score(df_new):
    amap = bundle.get("alias_map")
    if amap:  # columns were aliased at train time -> rename incoming real names to aliases
        df_new = df_new.rename(columns=amap["forward"])
    d = bundle["cleaning_artifacts"].transform(df_new)
    feats = bundle["selected_raw_features"]
    num = [c for c in feats if c in d.select_dtypes(include=[np.number]).columns]
    cat = [c for c in feats if c not in num]
    X = bundle["preprocessor"].transform(d[num + cat])
    return bundle["model"].predict_proba(X)[:, 1]
print("scorer ready; threshold =", bundle["optimal_threshold"])

scorer ready; threshold = 0.6530009829366847


In [ ]:
# ── 7) Double Validation Agent — runs AFTER everything (training + eval + charts) ─────────────
# Deterministic checks (reliable bug/anomaly detection) + optional LLM second opinion (grounded,
# anti-hallucination). WARN-only: it never blocks. Saves validation_report.json/.md to the run
# folder and prints the GREEN/AMBER/RED verdict + each finding.
import os
from app.validation_agent import validate_run

_report = validate_run(
    run_dir=result.run_dir,
    result=result, prepared=prepared, model=model, summary=summary,
    cached_llm=cached_llm,                       # LLM 2nd opinion when USE_LLM; skipped otherwise
    context={"use_llm": USE_LLM,
             "tavily_key_set": bool(os.environ.get("TAVILY_API_KEY"))},
    verbose=True,
)
print(f"\nValidation report saved to: {result.run_dir}/validation_report.md")

In [6]:
# ── 6) Cost & Token Summary — saved as CSV in the run artifact folder ─────────
import json, csv
from pathlib import Path

_cost_path = Path(result.run_dir) / "cost_summary.json"
if _cost_path.exists():
    _c = json.loads(_cost_path.read_text(encoding="utf-8"))
else:
    _c = {"input_tokens": 0, "cached_tokens": 0,
          "output_tokens": 0, "thinking_tokens": 0, "total_tokens": 0, "model": "—"}

_row = {
    "Use Case":         OBJECTIVE,
    "Model":            _c.get("model", cached_llm.model_id if USE_LLM else "—"),
    "Iteration ID":     Path(result.run_dir).name,
    "Prompt Tokens":    _c.get("input_tokens", 0),
    "Cached Tokens":    _c.get("cached_tokens", 0),
    "Output Tokens":    _c.get("output_tokens", 0),
    "Thinking Tokens":  _c.get("thinking_tokens", 0),
    "Total Tokens":     _c.get("total_tokens", 0),
}

# ── Print summary table in output cell ───────────────────────────────────────
_W = 64
print("═" * _W)
print("  TOKEN USAGE SUMMARY")
print("─" * _W)
for k, v in _row.items():
    print(f"  {k:<20}  {v}")
print("═" * _W)

# ── Write / append to a shared cost_log.csv in the artifacts root ─────────────
_artifacts_root = Path(result.run_dir).parent   # e.g. artifacts/runs/
_csv_path = _artifacts_root / "cost_log.csv"
_write_header = not _csv_path.exists()
with open(_csv_path, "a", newline="", encoding="utf-8") as _f:
    _w = csv.DictWriter(_f, fieldnames=list(_row.keys()))
    if _write_header:
        _w.writeheader()
    _w.writerow(_row)

print(f"\n  ✓ Row appended to:  {_csv_path}")

════════════════════════════════════════════════════════════════
  TOKEN USAGE SUMMARY
────────────────────────────────────────────────────────────────
  Use Case              propensity model
  Model                 gemini-2.5-flash-lite
  Iteration ID          2026-06-19_14-49-22
  Prompt Tokens         494
  Cached Tokens         0
  Output Tokens         1050
  Thinking Tokens       0
  Total Tokens          1544
════════════════════════════════════════════════════════════════

  ✓ Row appended to:  artifacts\runs\cost_log.csv
